<a href="https://colab.research.google.com/github/Saad-cpp/Data-Science-Internship/blob/main/GraphQA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install neo4j langchain langchain-experimental langchain-core langchain-community langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.6/296.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.1/208.1 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 404.4/404.4 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.8/160.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 760.0/760.0 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.0/78.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3

In [ ]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.5/106.5 kB 3.7 MB/s eta 0:00:00


In [ ]:
from langchain_community.graphs import Neo4jGraph

In [ ]:
from google.colab import userdata
gem = userdata.get('GEMINI-KEY')

In [ ]:
from google.colab import userdata
neo_url = userdata.get('neo-url')
neo_pass = userdata.get('neo-pass')
neo_user = userdata.get('neo-user')
groq_key = userdata.get('grqo_key')

In [ ]:
graph = Neo4jGraph(
    url=neo_url,
    username=neo_user,
    password=neo_pass
)

In [ ]:
print(graph.schema)

Node properties:
Patient {Id: STRING, BIRTHDATE: DATE_TIME, FIRST: STRING, MIDDLE: STRING, LAST: STRING, BIRTHPLACE: STRING, ADDRESS: STRING, INCOME: INTEGER, ZIP: INTEGER, LAT: FLOAT, LON: FLOAT, RACE: STRING, ETHNICITY: STRING, GENDER: STRING, HEALTHCARE_EXPENSES: FLOAT, HEALTHCARE_COVERAGE: FLOAT, MARITAL: STRING, DEATHDATE: DATE_TIME}
State {STATE: STRING}
City {CITY: STRING}
County {COUNTY: STRING}
Allergy {CODE: INTEGER, DESCRIPTION: STRING, TYPE: STRING, CATEGORY: STRING}
Careplan {CODE: INTEGER, DESCRIPTION: STRING}
Condition {CODE: INTEGER, DESCRIPTION: STRING}
Immunization {CODE: INTEGER, DESCRIPTION: STRING}
Payer {Id: STRING, NAME: STRING, OWNERSHIP: STRING, AMOUNT_COVERED: FLOAT, AMOUNT_UNCOVERED: FLOAT, REVENUE: FLOAT, COVERED_ENCOUNTERS: INTEGER, UNCOVERED_ENCOUNTERS: INTEGER, COVERED_MEDICATIONS: INTEGER, UNCOVERED_MEDICATIONS: INTEGER, COVERED_PROCEDURES: INTEGER, UNCOVERED_PROCEDURES: INTEGER, COVERED_IMMUNIZATIONS: INTEGER, UNCOVERED_IMMUNIZATIONS: INTEGER, UNIQUE_CU

In [ ]:
from langchain.chains import GraphCypherQAChain
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

# llm = ChatGoogleGenerativeAI(
#     model='gemini-1.5-flash',
#     api_key=gem,
#     temperature = 0
# )

llm = ChatGroq(
    model='llama-3.1-70b-versatile',
    api_key=groq_key,
    temperature = 0.0
)

In [ ]:
examples = [
    {
        "question": "How many allergies are of category food?",
        "query": "MATCH (a:Allergy) WHERE toLower(a.CATEGORY) CONTAINS 'food' RETURN count(a)",
    },
    {
        "question": "Which patient has gotten Hematologic disorder medication procedure?",
        "query": """MATCH (p:Patient)-[:GOT_PROCEDURE]->(proc:Procedure)
        WHERE toLower(proc.DESCRIPTION) CONTAINS toLower("Hematologic disorder medication")
        RETURN p.Id""",
    },
    {
        "question": "Show me base cost of immunizations for patient with first name kelly",
        "query": """MATCH (p:Patient)-[g:GOT_IMMUNIZATION]->(i:Immunization)
        WHERE toLower(p.FIRST) CONTAINS 'kelly'
        RETURN i.DESCRIPTION, g.BASE_COST""",
    },
    {
        "question": "what allergies do patients have who got Allergy screening test procedure?",
        "query": """MATCH (p:Patient)-[:GOT_PROCEDURE]->(proc:Procedure)
        WHERE toLower(proc.DESCRIPTION) CONTAINS toLower('Allergy screening test')
        MATCH (p)-[:ALLERGIC_TO]->(a:Allergy)
        RETURN DISTINCT a.DESCRIPTION""",
    },
    {
        "question": "How many patients have ambulatory encounter?",
        "query": """MATCH (p:Patient)<-[:HAS_ENCOUNTER]-(e:Encounter)
        WHERE toLower(e.ENCOUNTER_TYPE) CONTAINS 'ambulatory'
        RETURN count(p)""",
    },
    {
        "question": "How many patients do not have insurance?",
        "query": """MATCH (p:Patient)-[:HAS_PAYER]->(py:Payer)
        WHERE py.NAME = "NO_INSURANCE"
        RETURN count(p)""",
    },
    {
        "question": "Show me condtitions of patients living in Norfolk county?",
        "query": """MATCH (p:Patient)-[:LIVES_IN]->(co:County)
        WHERE toLower(co.COUNTY) CONTAINS toLower('Norfolk')
        MATCH (p)-[:HAS_CONDITION]->(c:Condition)
        RETURN c.DESCRIPTION""",
    },
    {
        "question": "Show me population of cities",
        "query": """MATCH (c:City)<-[:IS_IN]-(:County)<-[:LIVES_IN]-(p:Patient)
        RETURN c.CITY, count(p)
        ORDER BY count(p) DESC"""
      },
    {
        "question": "Show patients with wound in careplan and reason for that careplan.",
        "query": """MATCH (p:Patient)-[h:HAS_PLAN]->(c:Careplan)
        WHERE toLower(c.DESCRIPTION) CONTAINS toLower('wound')
        RETURN p.FIRST, h.REASONDESCRIPTION"""
      },
    {
        "question": "Show procedure of patients with cancer in condition.",
        "query": """MATCH (p:Patient)-[:HAS_CONDITION]->(c:Condition)
        WHERE toLower(c.DESCRIPTION) CONTAINS toLower('cancer')
        MATCH (p)-[:GOT_PROCEDURE]->(pro:Procedure)
        RETURN p.FIRST, c.DESCRIPTION, pro.DESCRIPTION"""
      }
]

In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 603.0/603.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.7/94.7 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.2/93.2 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.2/13.2 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.5/52.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.0/64.0 kB 6.2 MB/s eta 0:00:

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from langchain_community.vectorstores import Chroma

vector_store = Chroma(collection_name="cypher_example")
embedding_provider = GoogleGenerativeAIEmbeddings(model='models/text-embedding-004', google_api_key=gem)
example_selector = SemanticSimilarityExampleSelector.from_examples(
    examples, embedding_provider, vector_store, k=5, input_keys= ["question"]
)

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

example_prompt = PromptTemplate.from_template(
    "User input: {question}\ncypher: {query}"
)
prompt = FewShotPromptTemplate(
    example_selector=example_selector,
    example_prompt=example_prompt,
    prefix="""You are an expert in translating natural language questions into Cypher queries for Neo4j. Follow these guidelines:
    - Use only the relationship types and properties explicitly provided in the question and schema.
    - Always apply `toLower()` when comparing strings, and use partial string matching (e.g., `CONTAINS`) where appropriate.
    - Always validate the generated Cypher queries to ensure they are correct.
    - Do not include any explanation or extra text in your output. Only return the Cypher query, without any additional words, quotes, or prefixes (such as 'query', 'user input').
    - Refer to the provided schema information carefully to understand node labels, relationship types, and properties. Use this schema information to ensure accuracy.


    Here is the schema information
    Schema: {schema}


    Below are several examples of questions and their corresponding Cypher queries that demonstrate how to form your query based on the user input.""",
    suffix="User Input: {question}\n cypher: ",
    input_variables=["question", "schema"],
)


In [ ]:
# chain = GraphCypherQAChain.from_llm(llm=llm,
#     validate_cypher=True,
#     graph=graph,
#     verbose=True,
#     return_intermediate_steps=True,
#     # return_direct=True,
#     cypher_prompt=prompt,
#     allow_dangerous_requests=True)

## Printing results

In [ ]:
# print(chain.invoke({"query": """show me procedure descriptions
# of patients who have peanut in allergy description.
# I need only procedure and allergy"""})['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Patient)-[:ALLERGIC_TO]->(a:Allergy)
WHERE toLower(a.DESCRIPTION) CONTAINS toLower('peanut')
MATCH (p)-[:GOT_PROCEDURE]->(proc:Procedure)
RETURN DISTINCT proc.DESCRIPTION, a.DESCRIPTION

> Finished chain.
[{'proc.DESCRIPTION': 'Medication Reconciliation (procedure)', 'a.DESCRIPTION': 'Peanut (substance)'}, {'proc.DESCRIPTION': 'Dental consultation and report (procedure)', 'a.DESCRIPTION': 'Peanut (substance)'}, {'proc.DESCRIPTION': 'Dental X-ray bitewing (procedure)', 'a.DESCRIPTION': 'Peanut (substance)'}, {'proc.DESCRIPTION': 'Patient referral for dental care (procedure)', 'a.DESCRIPTION': 'Peanut (substance)'}, {'proc.DESCRIPTION': 'Dental care (regime/therapy)', 'a.DESCRIPTION': 'Peanut (substance)'}, {'proc.DESCRIPTION': 'Removal of supragingival plaque and calculus from all teeth using dental instrument (procedure)', 'a.DESCRIPTION': 'Peanut (substance)'}, {'proc.DESCRIPTION': 'Removal of subgingival plaque a

In [ ]:
# print(chain.invoke({"query": """show me payer information
# of patients who have ambulatory in encounterclass.
# Return payer name only"""})['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Patient)<-[:GOT_ENCOUNTER]-(e:Encounter)
WHERE toLower(e.ENCOUNTERCLASS) CONTAINS 'ambulatory'
MATCH (p)-[:HAS_PAYER]->(py:Payer)
RETURN py.NAME

> Finished chain.
[{'py.NAME': 'Medicaid'}, {'py.NAME': 'NO_INSURANCE'}, {'py.NAME': 'Humana'}, {'py.NAME': 'Medicaid'}, {'py.NAME': 'Medicaid'}, {'py.NAME': 'Medicare'}, {'py.NAME': 'Medicaid'}, {'py.NAME': 'Aetna'}, {'py.NAME': 'Anthem'}, {'py.NAME': 'Medicaid'}]


In [ ]:
# print(chain.invoke({"query": """show me 3 most common payer information."""})['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Payer) RETURN p.NAME, p.OWNERSHIP, p.AMOUNT_COVERED, p.AMOUNT_UNCOVERED, p.REVENUE, p.COVERED_ENCOUNTERS, p.UNCOVERED_ENCOUNTERS, p.COVERED_MEDICATIONS, p.UNCOVERED_MEDICATIONS, p.COVERED_PROCEDURES, p.UNCOVERED_PROCEDURES, p.COVERED_IMMUNIZATIONS, p.UNCOVERED_IMMUNIZATIONS, p.UNIQUE_CUSTOMERS, p.QOLS_AVG, p.MEMBER_MONTHS ORDER BY p.UNIQUE_CUSTOMERS DESC LIMIT 3


> Finished chain.
[{'p.NAME': 'Medicaid', 'p.OWNERSHIP': 'GOVERNMENT', 'p.AMOUNT_COVERED': 15381861.25, 'p.AMOUNT_UNCOVERED': 344533.13, 'p.REVENUE': 132580.0, 'p.COVERED_ENCOUNTERS': 4116, 'p.UNCOVERED_ENCOUNTERS': 0, 'p.COVERED_MEDICATIONS': 1814, 'p.UNCOVERED_MEDICATIONS': 0, 'p.COVERED_PROCEDURES': 9408, 'p.UNCOVERED_PROCEDURES': 0, 'p.COVERED_IMMUNIZATIONS': 1494, 'p.UNCOVERED_IMMUNIZATIONS': 0, 'p.UNIQUE_CUSTOMERS': 54, 'p.QOLS_AVG': 0.9383050195709091, 'p.MEMBER_MONTHS': 13344}, {'p.NAME': 'NO_INSURANCE', 'p.OWNERSHIP': 'NO_INSURANCE', 'p.AMOUNT_CO

In [ ]:
# print(chain.invoke({"query": """show me 3 most common immunizations
# in patients with allergy description that contains substance. Return allergy description, immunization description and count only."""})['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Patient)-[:GOT_IMMUNIZATION]->(imm:Immunization)
MATCH (p)-[:ALLERGIC_TO]->(a:Allergy)
WHERE toLower(a.DESCRIPTION) CONTAINS toLower('substance')
WITH a.DESCRIPTION AS Allergy_Description, imm.DESCRIPTION AS Immunization_Description, COUNT(*) AS Count
ORDER BY Count DESC
RETURN Allergy_Description, Immunization_Description, Count
LIMIT 3

> Finished chain.
[{'Allergy_Description': 'Allergy to substance (finding)', 'Immunization_Description': 'Influenza  seasonal  injectable  preservative free', 'Count': 15}, {'Allergy_Description': 'Allergy to substance (finding)', 'Immunization_Description': 'Td (adult)  5 Lf tetanus toxoid  preservative free  adsorbed', 'Count': 10}, {'Allergy_Description': 'Grass pollen (substance)', 'Immunization_Description': 'Influenza  seasonal  injectable  preservative free', 'Count': 9}]


In [ ]:
# print(chain.invoke({"query": """show me first names of patients who have same condition as patient with first name Lily490. Return only names of other patients and conditions"""})['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p1:Patient {FIRST: "Lily490"})-[:HAS_CONDITION]->(c:Condition)<-[:HAS_CONDITION]-(p2:Patient)
WHERE p1.Id <> p2.Id
RETURN p2.FIRST, c.DESCRIPTION


> Finished chain.
[{'p2.FIRST': 'Aurelia213', 'c.DESCRIPTION': 'Medication review due (situation)'}, {'p2.FIRST': 'Elwood28', 'c.DESCRIPTION': 'Medication review due (situation)'}, {'p2.FIRST': 'Claudia969', 'c.DESCRIPTION': 'Medication review due (situation)'}, {'p2.FIRST': 'Jen355', 'c.DESCRIPTION': 'Medication review due (situation)'}, {'p2.FIRST': 'Wallace647', 'c.DESCRIPTION': 'Medication review due (situation)'}, {'p2.FIRST': 'Harris789', 'c.DESCRIPTION': 'Medication review due (situation)'}, {'p2.FIRST': 'Glynis780', 'c.DESCRIPTION': 'Medication review due (situation)'}, {'p2.FIRST': 'Arnetta705', 'c.DESCRIPTION': 'Medication review due (situation)'}, {'p2.FIRST': 'Alvin56', 'c.DESCRIPTION': 'Medication review due (situation)'}, {'p2.FIRST': 'Maurita219', 'c.DESCRI

In [ ]:
# print(chain.invoke({"query": """show me most 3 common immunization in county named Norfolk county. Return immunization and count"""})['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Patient)-[:LIVES_IN]->(co:County)
WHERE toLower(co.COUNTY) CONTAINS toLower('Norfolk')
MATCH (p)-[:GOT_IMMUNIZATION]->(i:Immunization)
RETURN i.DESCRIPTION, count(i) ORDER BY count(i) DESC LIMIT 3


> Finished chain.
[{'i.DESCRIPTION': 'Influenza  seasonal  injectable  preservative free', 'count(i)': 9}, {'i.DESCRIPTION': 'Td (adult)  5 Lf tetanus toxoid  preservative free  adsorbed', 'count(i)': 6}, {'i.DESCRIPTION': 'HPV  quadrivalent', 'count(i)': 4}]


In [ ]:
# print(chain.invoke({"query": """show me county name
# of patient who have therapy in careplan description.
# Return county name only with patient first name and careplan and nothing else"""})['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Patient)-[:HAS_PLAN]->(cp:Careplan)
WHERE toLower(cp.DESCRIPTION) CONTAINS toLower('therapy')
MATCH (p)-[:LIVES_IN]->(c:County)
RETURN c.COUNTY, p.FIRST, cp.DESCRIPTION


> Finished chain.
[{'c.COUNTY': 'Hampden County', 'p.FIRST': 'Aurelia213', 'cp.DESCRIPTION': 'Respiratory therapy'}, {'c.COUNTY': 'Norfolk County', 'p.FIRST': 'Wallace647', 'cp.DESCRIPTION': 'Respiratory therapy'}, {'c.COUNTY': 'Hampshire County', 'p.FIRST': 'Glynis780', 'cp.DESCRIPTION': 'Respiratory therapy'}, {'c.COUNTY': 'Middlesex County', 'p.FIRST': 'Lizabeth515', 'cp.DESCRIPTION': 'Respiratory therapy'}, {'c.COUNTY': 'Middlesex County', 'p.FIRST': 'Rowena386', 'cp.DESCRIPTION': 'Respiratory therapy'}, {'c.COUNTY': 'Essex County', 'p.FIRST': 'Alina705', 'cp.DESCRIPTION': 'Respiratory therapy'}, {'c.COUNTY': 'Middlesex County', 'p.FIRST': 'Johnathon489', 'cp.DESCRIPTION': 'Respiratory therapy'}, {'c.COUNTY': 'Berkshire County', 'p.FIRST': 'Da

In [ ]:
# print(chain.invoke({"query": """Which conditions first 1 patient who lives in Milton city have. Return patient name and condition"""}))



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (p:Patient)-[:LIVES_IN]->(co:County)-[:IS_IN]->(ci:City)
WHERE toLower(ci.CITY) CONTAINS toLower('Milton')
WITH p LIMIT 1
MATCH (p)-[:HAS_CONDITION]->(c:Condition)
RETURN p.FIRST, p.LAST, c.DESCRIPTION 


> Finished chain.
{'query': 'Which conditions first 1 patient who lives in Milton city have. Return patient name and condition', 'result': [{'p.FIRST': 'Elwood28', 'p.LAST': 'Wyman904', 'c.DESCRIPTION': 'Loss of teeth (disorder)'}, {'p.FIRST': 'Elwood28', 'p.LAST': 'Wyman904', 'c.DESCRIPTION': 'Risk activity involvement (finding)'}, {'p.FIRST': 'Elwood28', 'p.LAST': 'Wyman904', 'c.DESCRIPTION': 'Received higher education (finding)'}, {'p.FIRST': 'Elwood28', 'p.LAST': 'Wyman904', 'c.DESCRIPTION': 'Medication review due (situation)'}, {'p.FIRST': 'Elwood28', 'p.LAST': 'Wyman904', 'c.DESCRIPTION': 'Prediabetes'}, {'p.FIRST': 'Elwood28', 'p.LAST': 'Wyman904', 'c.DESCRIPTION': 'Anemia (disorder)'}, {'p.FIRST': 'Elwood28',

In [ ]:
# print(chain.invoke({"query": """List 5 distinct cities with total number of patients living in them and the most common condition present in patients of that city"""})['result'])



> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (c:City)<-[:IS_IN]-(co:County)<-[:LIVES_IN]-(p:Patient)
WHERE toLower(co.COUNTY) CONTAINS toLower('Norfolk')
WITH c, COLLECT(DISTINCT p) AS patients
OPTIONAL MATCH (p)-[:HAS_CONDITION]->(c:Condition)
GROUP BY c
ORDER BY SIZE(patients) DESC
RETURN c.CITY, SIZE(patients), COLLECT(DISTINCT c.DESCRIPTION) AS conditions
LIMIT 5


CypherSyntaxError: {code: Neo.ClientError.Statement.SyntaxError} {message: Invalid input 'GROUP': expected a graph pattern, 'FOREACH', ',', 'ORDER BY', 'CALL', 'CREATE', 'LOAD CSV', 'DELETE', 'DETACH', 'FINISH', 'INSERT', 'LIMIT', 'MATCH', 'MERGE', 'NODETACH', 'OFFSET', 'OPTIONAL', 'REMOVE', 'RETURN', 'SET', 'SKIP', 'UNION', 'UNWIND', 'USE', 'USING', 'WHERE', 'WITH' or <EOF> (line 5, column 1 (offset: 206))
"GROUP BY c"
 ^}

## Gradio

In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.8/319.8 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 436.6/436.6 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 12.5 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 13.1
    Uninstalling websockets-13.1:
      Successfully uninstalled websockets-13.1
  Attempting uninstall: markupsafe
    Found existing installation: MarkupSafe 3.0.1
    Uninstalling MarkupSafe-3.0.1:
      Successfully uninstalled MarkupSafe-3.0.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.24.7
    Uninstalling huggingface-hub-0.24.7:
      Successfully uninstalled huggingface-hub-0.24.7


In [ ]:
from neo4j.exceptions import Neo4jError

def gr_func(query):
  chain = llm
  pr = prompt.format(question=query, schema=graph.schema)
  que = chain.invoke(pr).content
  print(que,end='\n\n\n')

  max_retries = 3
  attempt = 0
  while attempt < max_retries:

    try:
      result = graph.query(que)

      if len(result) == 0:
        if attempt < max_retries:
          err = f"{que}\n This query gives no result.\n. Validate its semantics and correct this query without any explaination.\n{pr}"
          que = chain.invoke(err).content
          print(que,end='\n')
          print('\33[93m'+"Retrying"+'\33[0m')
          attempt += 1
        else:
          return "No results found. Try again"
      answer = ""
      for k in result[0].keys():
        answer += k + "\t"
      answer += "\n"
      for rec in result:
        for i in rec.values():
          answer += str(i) + ",\t\t"
        answer += "\n\n"
      return answer

    except Neo4jError as e:
      attempt += 1
      print('\33[41m'+"ERROR OCCURED"+'\33[0m')
      err = f"{que}\n This query generates following error {e}.\n. Correct this query without any explaination.\n{pr}"
      if attempt < max_retries:
        que = chain.invoke(err).content
        print(que,end='\n')
        print('\33[93m'+"Retrying"+'\33[0m')
      else:
        return "ERROR OCCURED. TRY AGAIN"

In [ ]:
import gradio as gr

gr.Interface(fn=gr_func, inputs="text", outputs="text").launch(debug=True)

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4872e072d6f0a80066.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


MATCH (p:Patient)-[:HAS_CONDITION]->(c:Condition)
        WHERE toLower(c.DESCRIPTION) CONTAINS toLower('cancer')
        MATCH (p)-[:IS_ON]->(m:Medications)
        RETURN m.DESCRIPTION


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://4872e072d6f0a80066.gradio.live


show patients which have cancer and show their care plan. Return patient first name, county name, list of careplan description and single condition description

show distinct first names of alive patients which have same condition as dead patients. Return first name and common distinct condition list

show ages of patients who have died. Show first name and age in years

In [ ]:
query = """show me number of patients living in each city. Return top 5 cities by population"""

pr = prompt.format(question=query, schema=graph.schema)
que = chain.invoke(pr).content
print(que)

MATCH (c:City)<-[:IS_IN]-(:County)<-[:LIVES_IN]-(p:Patient)
RETURN c.CITY, count(p) AS population
ORDER BY population DESC
LIMIT 5


In [ ]:
chain.invoke({"query": """show me number of patients living in each city. Return top 5 cities by population"""})['result']



> Entering new GraphCypherQAChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
You are an expert in translating natural language questions into Cypher queries for Neo4j. Follow these guidelines:
    - Use only the relationship types and properties explicitly provided in the question and schema.
    - Always apply `toLower()` when comparing strings, and use partial string matching (e.g., `CONTAINS`) where appropriate.
    - Always validate the generated Cypher queries to ensure they are correct.
    - Do not include any explanation or extra text in your output. Only return the Cypher query, without any additional words, quotes, or prefixes (such as 'query', 'user input').
    - Refer to the provided schema information carefully to understand node labels, relationship types, and properties. Use this schema information to ensure accuracy.


    Here is the schema information
    Schema: Node properties are the following:
Patient {Id: STRING, BIRTHDATE: DATE_TIME,

'Acton, Waltham, Concord, Shirley, and Tewksbury each have 27 patients. \n'

In [ ]:
import langchain
langchain.globals.get_verbose()

True

In [ ]:
print('\33[93m'+"HELLO")

HELLO
